# Notebook 8 — Naive Bayes
### Sprint 7 | Machine Learning Fundamentals for AI/ML Engineers


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])
df_encoded = df.copy()
for col in encode_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

X = df_encoded.drop(columns=['customerID', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")


Train: (5634, 19) | Test: (1409, 19)


/tmp/ipykernel_677/2007514712.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])


---
## Algorithm Documentation

### What is the algorithm?
A probabilistic classifier built directly on Bayes' Theorem — computes the probability
of each class GIVEN the observed features, using each feature's individual relationship
with the class.

### Problem Type
**Classification.**

### Concept — Bayes' Theorem & Conditional Probability
$$P(\text{Class} \mid \text{Features}) = \frac{P(\text{Features} \mid \text{Class}) \cdot P(\text{Class})}{P(\text{Features})}$$

In words: the probability of churn GIVEN what we observe about a customer is
proportional to (how likely these observations are IF the customer churns) times (how
common churn is overall).

### Why "Naive"?
The algorithm makes a deliberately simplifying — "naive" — assumption: it assumes every
feature is CONDITIONALLY INDEPENDENT of every other feature, given the class. In
reality, this is almost never exactly true (e.g., `MonthlyCharges` and `TotalCharges`
are strongly correlated, Sprint 4, Notebook 6's VIF~8.1 finding) — the algorithm ignores
this and treats each feature's contribution as if it were independent anyway. Despite
this unrealistic assumption, Naive Bayes often still performs surprisingly well in
practice.

### Mathematical Intuition
Because of the independence assumption, the joint probability of ALL features given the
class simplifies to a simple PRODUCT of each individual feature's probability given the
class — dramatically reducing the computation needed compared to modeling the full joint
distribution.

### Example


In [2]:
# A tiny illustrative example: does a high MonthlyCharges alone shift the churn probability?
prior_churn = y_train.mean()
high_charge_customers = X_train[X_train['MonthlyCharges'] > X_train['MonthlyCharges'].median()]
likelihood_high_charge_given_churn = y_train[high_charge_customers.index].mean()

print(f"P(Churn) [prior]                          : {prior_churn:.3f}")
print(f"P(Churn | MonthlyCharges > median)         : {likelihood_high_charge_given_churn:.3f}")
print("This shift from prior to posterior, driven by ONE piece of evidence, is exactly")
print("the mechanism Naive Bayes formalizes and combines across ALL features at once.")


P(Churn) [prior]                          : 0.265
P(Churn | MonthlyCharges > median)         : 0.351
This shift from prior to posterior, driven by ONE piece of evidence, is exactly
the mechanism Naive Bayes formalizes and combines across ALL features at once.


### Implementation — Gaussian, Multinomial, and Bernoulli Variants


In [3]:
# Gaussian Naive Bayes: assumes each numeric feature is normally distributed within each class
gnb = GaussianNB()
gnb.fit(X_train, y_train)
gnb_acc = accuracy_score(y_test, gnb.predict(X_test))
print(f"Gaussian Naive Bayes test accuracy: {gnb_acc:.4f}")

# Bernoulli Naive Bayes: designed for binary (0/1) features
binary_cols = [c for c in X.columns if X[c].nunique() == 2]
bnb = BernoulliNB()
bnb.fit(X_train[binary_cols], y_train)
bnb_acc = accuracy_score(y_test, bnb.predict(X_test[binary_cols]))
print(f"Bernoulli Naive Bayes (binary columns only) test accuracy: {bnb_acc:.4f}")


Gaussian Naive Bayes test accuracy: 0.7452
Bernoulli Naive Bayes (binary columns only) test accuracy: 0.7445


### Multinomial Naive Bayes — Text Classification Use Case
Multinomial NB is built for COUNT data — most classically, word counts in text
classification (e.g., spam detection, sentiment analysis). This dataset has no text
column (confirmed repeatedly since Sprint 4), so this variant is demonstrated on a small
illustrative text example instead, honestly separated from the real Telco results above.


In [4]:
illustrative_texts = [
    "great service love it", "terrible service want refund", "excellent support team",
    "worst experience ever", "happy with my plan", "cancel my subscription now"
]
illustrative_labels = [1, 0, 1, 0, 1, 0]   # 1 = positive, 0 = negative

vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(illustrative_texts)

mnb = MultinomialNB()
mnb.fit(X_text, illustrative_labels)

new_text = vectorizer.transform(["amazing service and support"])
prediction = mnb.predict(new_text)
print(f"Multinomial NB prediction for 'amazing service and support': {'positive' if prediction[0]==1 else 'negative'}")
print("(Illustrative only — the real Telco dataset has no text column for this variant.)")


Multinomial NB prediction for 'amazing service and support': positive
(Illustrative only — the real Telco dataset has no text column for this variant.)


### Parameters
- **`var_smoothing`** (Gaussian, default `1e-9`): a small value added to variance
  estimates to avoid numerical instability from zero-variance features.
- **`alpha`** (Multinomial/Bernoulli, default `1.0`): Laplace smoothing — prevents a
  zero probability for a feature value never seen in training (which would otherwise
  make the ENTIRE product zero, regardless of other evidence).

### Result & Interpretation


In [5]:
print(f"Gaussian NB accuracy  : {gnb_acc:.4f}")
print(f"Bernoulli NB accuracy  : {bnb_acc:.4f} (using only the {len(binary_cols)} binary columns)")
print(f"\nFor comparison, Logistic Regression (Notebook 6) scored: 0.7388 accuracy")


Gaussian NB accuracy  : 0.7452
Bernoulli NB accuracy  : 0.7445 (using only the 6 binary columns)

For comparison, Logistic Regression (Notebook 6) scored: 0.7388 accuracy


**Interpretation:** Naive Bayes's independence assumption is clearly violated by
this dataset (known multicollinearity, Sprint 4-6), yet the algorithm often remains
competitive — a useful, hands-on demonstration that a "wrong" assumption doesn't
necessarily doom a model's practical usefulness, especially as a fast baseline.

### Advantages
- Extremely fast to train and predict — no iterative optimization needed.
- Works well with high-dimensional data (e.g., text with thousands of word features).
- Requires relatively little training data to estimate reasonable probabilities.

### Limitations
- The independence assumption is almost always technically wrong, which can hurt
  accuracy when features are strongly correlated (as demonstrated by this dataset's
  known multicollinearity).
- Gaussian NB specifically assumes normally-distributed features — Sprint 4, Notebook 8
  confirmed NONE of this dataset's numeric features are actually normal, a real,
  acknowledged mismatch between the algorithm's assumption and the data.

### When to Use
Text classification (spam filtering, sentiment analysis), as a fast baseline before
trying more complex models, or when training data is limited.

### When Not to Use
When features are strongly correlated and that correlation carries real predictive
information the independence assumption would discard, or when the underlying feature
distributions clearly violate the chosen variant's assumption (e.g., Gaussian NB on
confirmed non-normal data, as is technically the case here).


---
## Summary

| Variant | Best For | This Dataset |
|---|---|---|
| Gaussian NB | Continuous, ideally normal features | Used on full feature set — some assumption mismatch given non-normal features |
| Bernoulli NB | Binary (0/1) features | Used on the dataset's binary-only columns |
| Multinomial NB | Count data (text) | Not applicable to Telco (no text) — demonstrated illustratively |

**Next notebook:** `08_Decision_Trees.ipynb` — a rule-based approach that requires no
distributional assumptions at all.
